Generate bounded RP and Kill Chain validation receipts

In [1]:
cd /workspace/backend && PYTHONPATH=. python scripts/validate_monitoring_shadow.py /mnt/shared-workspace/shared/phase_d_sprint/w3/monitoring_shadow_receipt.json

{
  "receipt": "/mnt/shared-workspace/shared/phase_d_sprint/w3/monitoring_shadow_receipt.json",
  "sha256": "830b7c639ed7bb85143948b7227b62491709c41bd37868486f3ba2699d3f3a1c",
  "deterministic": true,
  "forbidden_visible": [],
  "observations": {
    "rp_longitudinal_state": "EVALUATED",
    "rp_detected_signal_count": 2,
    "kc_longitudinal_state": "resistance_detected",
    "kc_active_signals": {
      "CA125_RISING": true,
      "CTDNA_MRD": true
    },
    "rp_missing_state": "INSUFFICIENT_EVIDENCE",
    "kc_intake_state": "NOT_EVALUATED"
  }
}


Audit Synapse ovarian entities and access rights

In [3]:
from pathlib import Path
import requests, json, hashlib, os, re
from datetime import datetime, timezone

OUT=Path('/mnt/shared-workspace/shared/mechanism_fit_audit/lane3_synapse'); OUT.mkdir(parents=True,exist_ok=True)
DATA=Path('/workspace/mechanism_fit_synapse'); DATA.mkdir(parents=True,exist_ok=True)
def ts(): return datetime.now(timezone.utc).isoformat().replace('+00:00','Z')
def save(name,obj):
    p=OUT/name; p.write_text(json.dumps(obj,indent=2,default=str)); return p

def receipt(r,url,at):
    return {'url':url,'timestamp':at,'status_code':r.status_code,'headers':{k:v for k,v in r.headers.items() if k.lower() in ['content-type','content-length','etag','location','last-modified']},'body':r.text[:5000]}

repo='https://repo-prod.prod.sagebase.org/repo/v1'; fileapi='https://repo-prod.prod.sagebase.org/file/v1'
token=os.getenv('SYNAPSE_AUTH_TOKEN'); headers={'Authorization':'Bearer '+token} if token else {}
ids=['syn25569736','syn26642505','syn26642506','syn33521770','syn52458614','syn39607857','syn39607858','syn39607860','syn1709982','syn7222066','syn27056172']
manifest={'lane':'Synapse ovarian/recurrent-HGSOC','started_at':ts(),'authenticated_token_present':bool(token),'entities':[]}

# Verify authentication only if the rotated secret is present; never print/store token.
if token:
    url=repo+'/userProfile'; at=ts(); r=requests.get(url,headers=headers,timeout=60); save('authentication_receipt.json',receipt(r,url,at)); manifest['authentication_status']=r.status_code
else:
    save('authentication_receipt.json',{'timestamp':ts(),'status':'NO_CONFIGURED_ROTATED_SECRET_VISIBLE','note':'No token value was read, printed, or persisted.'}); manifest['authentication_status']='NO_CONFIGURED_ROTATED_SECRET_VISIBLE'

for eid in ids:
    ent={'id':eid,'attempts':[]}
    # Entity metadata and calculated access endpoint.
    for suffix,name in [('', 'entity'),('/access','access')]:
        url=f'{repo}/entity/{eid}{suffix}'; at=ts()
        try:
            r=requests.get(url,headers=headers,timeout=60,allow_redirects=False); rec=receipt(r,url,at); save(f'{eid}_{name}_receipt.json',rec); ent['attempts'].append({'type':name,'status_code':r.status_code,'timestamp':at,'receipt':f'{eid}_{name}_receipt.json'})
            if r.status_code==200:
                obj=r.json(); save(f'{eid}_{name}.json',obj); ent[name]=obj
        except Exception as e:
            rec={'url':url,'timestamp':at,'status_code':None,'error':type(e).__name__+': '+str(e)[:500]}; save(f'{eid}_{name}_receipt.json',rec); ent['attempts'].append({'type':name,'status_code':None,'timestamp':at,'receipt':f'{eid}_{name}_receipt.json'})
    # Bundle includes restrictions/file handles where visible.
    url=f'{repo}/entity/{eid}/bundle2'; at=ts(); payload={'includeEntity':True,'includeAnnotations':True,'includeFileHandles':True,'includeRestrictionInformation':True,'includeAccessControlList':True,'includeBenefactorACL':True}
    try:
        r=requests.post(url,headers={**headers,'Content-Type':'application/json'},json=payload,timeout=60,allow_redirects=False); rec=receipt(r,url,at); save(f'{eid}_bundle_receipt.json',rec); ent['attempts'].append({'type':'bundle2','status_code':r.status_code,'timestamp':at,'receipt':f'{eid}_bundle_receipt.json'})
        if r.status_code==200:
            b=r.json(); save(f'{eid}_bundle.json',b); ent['bundle_summary']={'fileHandles':[{'id':x.get('id'),'fileName':x.get('fileName'),'contentSize':x.get('contentSize'),'contentMd5':x.get('contentMd5')} for x in b.get('fileHandles',[])],'restrictionInformation':b.get('restrictionInformation')}
            for fh in b.get('fileHandles',[]):
                hid=fh.get('id')
                if not hid: continue
                durl=f'{fileapi}/fileHandle/{hid}/url'; dat=ts(); rr=requests.get(durl,headers=headers,timeout=60,allow_redirects=False); drec=receipt(rr,durl,dat); save(f'{eid}_filehandle_{hid}_access_receipt.json',drec)
                ent.setdefault('download_attempts',[]).append({'fileHandleId':hid,'status_code':rr.status_code,'timestamp':dat,'receipt':f'{eid}_filehandle_{hid}_access_receipt.json'})
    except Exception as e:
        rec={'url':url,'timestamp':at,'status_code':None,'error':type(e).__name__+': '+str(e)[:500]}; save(f'{eid}_bundle_receipt.json',rec); ent['attempts'].append({'type':'bundle2','status_code':None,'timestamp':at,'receipt':f'{eid}_bundle_receipt.json'})
    manifest['entities'].append(ent)

manifest['completed_at']=ts(); save('lane3_manifest.json',manifest)
print(json.dumps({'entities':len(ids),'metadata_accessed':sum(1 for e in manifest['entities'] if 'entity' in e),'bundles_accessed':sum(1 for e in manifest['entities'] if 'bundle_summary' in e),'token_present':bool(token),'output':str(OUT)},indent=2))

{
  "entities": 11,
  "metadata_accessed": 10,
  "bundles_accessed": 10,
  "token_present": false,
  "output": "/mnt/shared-workspace/shared/mechanism_fit_audit/lane3_synapse"
}


Traverse MSK-SPECTRUM and verify Synapse download rights

In [5]:
from pathlib import Path
import requests,json,os,hashlib
from datetime import datetime,timezone
OUT=Path('/mnt/shared-workspace/shared/mechanism_fit_audit/lane3_synapse')
def ts(): return datetime.now(timezone.utc).isoformat().replace('+00:00','Z')
def save(name,obj): (OUT/name).write_text(json.dumps(obj,indent=2,default=str))
def rec(r,url,at): return {'url':url,'timestamp':at,'status_code':r.status_code,'headers':{k:v for k,v in r.headers.items() if k.lower() in ['content-type','content-length','etag','location','last-modified']},'body':r.text[:4000]}
base='https://repo-prod.prod.sagebase.org/repo/v1'; fileapi='https://repo-prod.prod.sagebase.org/file/v1'; token=os.getenv('SYNAPSE_AUTH_TOKEN'); hdr={'Authorization':'Bearer '+token} if token else {}
manifest=json.loads((OUT/'lane3_manifest.json').read_text())
# Correct permission checks and direct file-handle checks for named IDs.
for ent in manifest['entities']:
 eid=ent['id']; m=ent.get('entity',{})
 for atype in ['READ','DOWNLOAD']:
  url=f'{base}/entity/{eid}/access?accessType={atype}'; at=ts(); r=requests.get(url,headers=hdr,timeout=60); save(f'{eid}_access_{atype}_receipt.json',rec(r,url,at)); ent.setdefault('permission_checks',[]).append({'accessType':atype,'status_code':r.status_code,'timestamp':at,'receipt':f'{eid}_access_{atype}_receipt.json'})
 hid=m.get('dataFileHandleId')
 if hid:
  url=f'{fileapi}/fileHandle/{hid}/url'; at=ts(); r=requests.get(url,headers=hdr,timeout=60,allow_redirects=False); save(f'{eid}_direct_filehandle_{hid}_receipt.json',rec(r,url,at)); ent['direct_download_check']={'fileHandleId':hid,'status_code':r.status_code,'timestamp':at,'receipt':f'{eid}_direct_filehandle_{hid}_receipt.json'}
# Walk visible MSK SPECTRUM hierarchy to depth 3.
front=[('syn25569736',0)]; seen=set(); discovered=[]
while front:
 parent,depth=front.pop(0)
 if parent in seen or depth>=4: continue
 seen.add(parent); token_page=None
 while True:
  payload={'parentId':parent,'includeTypes':['file','folder','table','entityview','dataset','materializedview'],'includeTotalChildCount':True}
  if token_page: payload['nextPageToken']=token_page
  url=f'{base}/entity/children'; at=ts(); r=requests.post(url,headers={**hdr,'Content-Type':'application/json'},json=payload,timeout=60)
  save(f'children_{parent}_{depth}_{len(discovered)}_receipt.json',rec(r,url,at))
  if r.status_code!=200: break
  obj=r.json()
  for c in obj.get('page',[]):
   item={**c,'parentId':parent,'depth':depth+1}; discovered.append(item)
   typ=str(c.get('type','')).lower()
   if 'folder' in typ: front.append((c.get('id'),depth+1))
  token_page=obj.get('nextPageToken')
  if not token_page: break
save('msk_spectrum_discovered_children.json',discovered)
# Inspect benchmark-relevant visible files and their exact permissions/download checks.
relevant=[]
for c in discovered:
 name=str(c.get('name','')).lower()
 if any(k in name for k in ['clinical','patient','sample','manifest','meta','mutation','.maf','cna','segment','outcome','therapy','treatment','wgs','rna']):
  eid=c.get('id'); item={'header':c}
  url=f'{base}/entity/{eid}'; at=ts(); r=requests.get(url,headers=hdr,timeout=60); save(f'discovered_{eid}_entity_receipt.json',rec(r,url,at)); item['entity_status']=r.status_code
  if r.status_code==200:
   m=r.json(); item['entity']=m; hid=m.get('dataFileHandleId')
   if hid:
    durl=f'{fileapi}/fileHandle/{hid}/url'; dat=ts(); rr=requests.get(durl,headers=hdr,timeout=60,allow_redirects=False); save(f'discovered_{eid}_download_receipt.json',rec(rr,durl,dat)); item['download_check']={'status_code':rr.status_code,'timestamp':dat,'receipt':f'discovered_{eid}_download_receipt.json','fileHandleId':hid}
  relevant.append(item)
save('msk_spectrum_relevant_entities.json',relevant)
manifest['msk_spectrum_children_discovered']=len(discovered); manifest['msk_spectrum_relevant_entities']=len(relevant); manifest['completed_at_extended']=ts(); save('lane3_manifest_extended.json',manifest)
print(json.dumps({'children':len(discovered),'relevant':[(x['header'].get('id'),x['header'].get('name'),x.get('entity_status'),x.get('download_check',{}).get('status_code')) for x in relevant],'known':[(e['id'],e.get('entity',{}).get('name'),e.get('entity',{}).get('parentId'),e.get('entity',{}).get('dataFileHandleId'),[(p['accessType'],p['status_code']) for p in e.get('permission_checks',[])],e.get('direct_download_check')) for e in manifest['entities']]},indent=2))

{
  "children": 2722,
  "relevant": [
    [
      "syn33521770",
      "Bulk WGS",
      200,
      null
    ],
    [
      "syn33521743",
      "scRNA",
      200,
      null
    ],
    [
      "syn52458611",
      "Bulk WGS",
      200,
      null
    ],
    [
      "syn52458609",
      "scRNA",
      200,
      null
    ],
    [
      "syn39607857",
      "cna.tsv",
      200,
      403
    ],
    [
      "syn39607858",
      "cohort.maf",
      200,
      403
    ],
    [
      "syn39607860",
      "segments.seg",
      200,
      403
    ],
    [
      "syn52458807",
      "segmentation",
      200,
      null
    ]
  ],
  "known": [
    [
      "syn25569736",
      "MSK SPECTRUM",
      "syn4489",
      null,
      [
        [
          "READ",
          200
        ],
        [
          "DOWNLOAD",
          200
        ]
      ],
      null
    ],
    [
      "syn26642505",
      "Multimodal integration of radiology, pathology, and genomics for prediction of response to PD-L1 

Inspect MSK-SPECTRUM hierarchy for clinical linkage layers

In [7]:
from pathlib import Path
import json,collections
p=Path('/mnt/shared-workspace/shared/mechanism_fit_audit/lane3_synapse/msk_spectrum_discovered_children.json')
d=json.loads(p.read_text())
for depth in sorted(set(x['depth'] for x in d)):
 print('\nDEPTH',depth,'count',sum(x['depth']==depth for x in d))
 for x in [y for y in d if y['depth']==depth][:120]: print(x.get('id'),x.get('name'),x.get('type'),x.get('parentId'))
print('\nkeyword matches')
for x in d:
 n=str(x.get('name','')).lower()
 if any(k in n for k in ['cohort','drug','therapy','treat','demog','clin','metadata','manifest','sample','patient','survival','outcome','response']): print(x)



DEPTH 1 count 7
syn33521770 Bulk WGS org.sagebionetworks.repo.model.table.Dataset syn25569736
syn33521762 H-E org.sagebionetworks.repo.model.table.Dataset syn25569736
syn33521771 MSK-IMPACT org.sagebionetworks.repo.model.table.Dataset syn25569736
syn66309959 McPherson et al. Nature 2025 org.sagebionetworks.repo.model.Folder syn25569736
syn52458607 Vazquez-Garcia, Uhlitz et al Nature 2022 org.sagebionetworks.repo.model.Folder syn25569736
syn33520881 mpIF org.sagebionetworks.repo.model.table.Dataset syn25569736
syn33521743 scRNA org.sagebionetworks.repo.model.table.Dataset syn25569736

DEPTH 2 count 6
syn66309960 IF org.sagebionetworks.repo.model.Folder syn66309959
syn52458611 Bulk WGS org.sagebionetworks.repo.model.Folder syn52458607
syn52458615 H-E org.sagebionetworks.repo.model.Folder syn52458607
syn52458614 MSK-IMPACT org.sagebionetworks.repo.model.Folder syn52458607
syn52434142 mpIF org.sagebionetworks.repo.model.Folder syn52458607
syn52458609 scRNA org.sagebionetworks.repo.model.F

Cross-review BriTROC field and access sufficiency

In [9]:
from pathlib import Path
import json
from datetime import datetime,timezone
root=Path('/mnt/shared-workspace/shared/mechanism_fit_audit'); z=json.loads((root/'lane1_britroc/zenodo_full_download.json').read_text())
review={'reviewer_lane':'Synapse worker cross-review','timestamp':datetime.now(timezone.utc).isoformat().replace('+00:00','Z'),'zenodo_archive_sha256':z['sha256'],'zenodo_members':len(z['zip_members']),'public_release_content':'Processed copy-number, short-variant, component-parameter, and per-sample clonality outputs; no standalone clinical/treatment/outcome dictionary was identified in the archive listing.','controlled_EGA_authorization':'Not tested because no EGA account secret/private key was visible in the execution environment.','verdict':'ACCESSIBLE_ENDPOINT_INCOMPATIBLE','exact_missing_field':'Public release lacks verified linked treatment-line, index-treatment-date, response/PFI event/censoring fields; controlled raw files are not authorized in this execution.'}
(root/'cross_review_britroc.json').write_text(json.dumps(review,indent=2));print(json.dumps(review,indent=2))

{
  "reviewer_lane": "Synapse worker cross-review",
  "timestamp": "2026-07-16T21:05:04.709832Z",
  "zenodo_archive_sha256": "fad81b4e8a0298b26fafc6e91a1e83c99856cafb9942eddd9efca89b2db071af",
  "zenodo_members": 1351,
  "public_release_content": "Processed copy-number, short-variant, component-parameter, and per-sample clonality outputs; no standalone clinical/treatment/outcome dictionary was identified in the archive listing.",
  "controlled_EGA_authorization": "Not tested because no EGA account secret/private key was visible in the execution environment.",
  "verdict": "ACCESSIBLE_ENDPOINT_INCOMPATIBLE",
  "exact_missing_field": "Public release lacks verified linked treatment-line, index-treatment-date, response/PFI event/censoring fields; controlled raw files are not authorized in this execution."
}
